**Final Project**

Student name: Khadija Bassiouni

**Introduction:** This project involves developing a sentiment analysis system that processes historical Twitter data from the Sentiment140 dataset. We will build a system to classify tweets into positive, negative, or neutral sentiments, through an end-to-end machine learning process.

**Data Collection and Preprocessing:**

**Step 1. Setting up the Environment:**

In [1]:
# Importing necessary libraries
import re
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [3]:
#  Unzipping the uploaded archive file
import zipfile

with zipfile.ZipFile("archive (2).zip", "r") as zip_ref:
    zip_ref.extractall(".")

In [4]:
# Loading Data
cols = ["target", "ids", "date", "flag", "user", "text"]
df = pd.read_csv("training.1600000.processed.noemoticon.csv", encoding="latin-1", names=cols)

In [5]:
# choosing a smaller sample for fast execution in Colab
df = df.sample(n=100000, random_state=42).reset_index(drop=True)

# Mapping targets: 0 = Negative, 4 = Positive -> Convert to 0 (Negative) and 1 (Positive)
df["target"] = df["target"].replace(4, 1)

In [6]:
# Cleaning Function
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", "", text, flags=re.MULTILINE) # Remove URLs
    text = re.sub(r"@\w+", "", text) # Remove mentions
    text = re.sub(r"[^a-zA-Z\s]", "", text) # Remove special characters and numbers
    text = re.sub(r"\s+", " ", text).strip() # Remove extra whitespace
    return text

print("Cleaning text data...")
df["cleaned_text"] = df["text"].apply(clean_text)

Cleaning text data...


**Model Training:**

In [7]:
#  Train/Validation Split
X_train, X_val, y_train, y_val = train_test_split(
    df["cleaned_text"], df["target"], test_size=0.2, random_state=42
)

#  Vectorization (TF-IDF)
vectorizer = TfidfVectorizer(max_features=50000, ngram_range=(1, 2))
X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec = vectorizer.transform(X_val)

In [8]:
# Model Training wit Logistic Regression
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

# Model Evaluation
y_pred = model.predict(X_val_vec)
print(f"Accuracy: {accuracy_score(y_val, y_pred):.4f}")
print(classification_report(y_val, y_pred))

# Saving Model and Vectorizer for Flask
joblib.dump(model, "sentiment_model.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")


Accuracy: 0.7893
              precision    recall  f1-score   support

           0       0.79      0.78      0.79      9995
           1       0.79      0.80      0.79     10005

    accuracy                           0.79     20000
   macro avg       0.79      0.79      0.79     20000
weighted avg       0.79      0.79      0.79     20000



['tfidf_vectorizer.pkl']

Model achieved an accuracy of 78.9% with balanced precision and F1-scores across both classes.

From this point, we will move to API Development and Containerization and MLOps Integration.

After Running the Container Locally and Testing It ( using Pycharm and docker desktop), we got a Positive with a confidence score of 0.887, pointing that our containerized Flask API is fully functional and successfully making predictions locally. Then The final step was to push our image to Docker Hub so it can be deployed or pulled by Kubernetes later.